# Produces a parent-child model first published in Mehl and Hill (2013)

The model in plan view


![alt_text](../../images/mehl_and_hill_2013.png)

### The usual preliminaries

In [ ]:
from pathlib import Path

import sys
import flopy
import flopy.utils.binaryfile as bf

import matplotlib.pyplot as plt
import numpy as np
from flopy.plot.styles import styles
from flopy.utils.lgrutil import Lgr
from modflow_devtools.misc import get_env, timed

In [ ]:
sys.path.insert(0, '../data/multi-model')
from sfr_data import *

### Set parameter values for later usage by the script

In [ ]:
# Name
example_name = "ex-gwf-lgr"

# Model units
length_units = "meters"
time_units = "days"

# Model parameters
nlayp = 3  # Number of layers in parent model
nrowp = 15  # Number of rows in parent model
ncolp = 15  # Number of columns in parent model
delrp = 102.94  # Parent model column width ($m$)
delcp = 68.63  # Parent model row width ($m$)
dum1 = 5  # Number of child model columns per parent model columns
dum2 = 5  # Number of child model rows per parent model rows
dum3 = 20.588  # Child model column width ($m$)
dum4 = 13.725  # Child model row width ($m$)
k11 = 1.0  # Horizontal hydraulic conductivity ($m/d$)
k33 = 1.0  # Vertical hydraulic conductivity ($m/d$)

# Additional model input preparation
# Time related variables
delrp = 1544.1 / ncolp
delcp = 1029.4 / nrowp
numdays = 1
perlen = [1] * numdays
nper = len(perlen)
nstp = [1] * numdays
tsmult = [1.0] * numdays

# Further parent model grid discretization
x = [round(x, 3) for x in np.linspace(50.0, 45.0, ncolp)]
topp = np.repeat(x, nrowp).reshape((15, 15)).T
z = [round(z, 3) for z in np.linspace(50.0, 0.0, nlayp + 1)]
botmp = [topp - z[len(z) - 2], topp - z[len(z) - 3], topp - z[0]]
idomainp = np.ones((nlayp, nrowp, ncolp), dtype=int)
idomainp[0:2, 6:11, 2:8] = 0  # Zero out where the child grid will reside
icelltype = [1, 0, 0]  # Water table resides in layer 1

# Solver settings
nouter, ninner = 100, 300
hclose, rclose, relax = 1e-7, 1e-6, 0.97

rbth = 1.5
rbhk = 0.1
man = 0.04
ustrf = 1.0
ndv = 0
pkdat = []
for i in np.arange(len(rlen)):
    if i < 8:
        rgrd = rgrd1
    else:
        rgrd = rgrd2
    ncon = len(connsp[i]) - 1
    pkdat.append(
        (
            i,
            sfrcells[i],
            rlen[i],
            rwid,
            rgrd,
            rbtp[i],
            rbth,
            rbhk,
            man,
            ncon,
            ustrf,
            ndv,
        )
    )

sfrspd = {0: [[0, "INFLOW", 40.0]]}

# Next, set up SFR input for the child model
# Start by defining the connections.  Set up with for loop since 1
# stream segment with all linear connections. Cheating a bit by the
# knowledge that there are 89 stream reaches in the child model.  This
# is known from the original model
connsc = []
for i in np.arange(89):
    if i == 0:
        connsc.append((i, -1 * (i + 1)))
    elif i == 88:
        connsc.append((i, i - 1))
    else:
        connsc.append((i, i - 1, -1 * (i + 1)))


rbthc = 1.5
rbhkc = 0.1
manc = 0.04
ustrfc = 1.0
ndvc = 0
pkdatc = []
for i in np.arange(len(rlenc)):
    nconc = len(connsc[i]) - 1
    pkdatc.append(
        (
            i,
            sfrcellsc[i],
            rlenc[i],
            rwidc,
            rgrdc,
            rbtpc[i],
            rbthc,
            rbhkc,
            manc,
            nconc,
            ustrfc,
            ndvc,
        )
    )

sfrspdc = {0: [[0, "INFLOW", 0.0]]}

### Model setup

Define functions to build models.  Start with the parent GWF model

In [ ]:
def add_parent_gwf_model(sim, gwfname):

    gwf = flopy.mf6.ModflowGwf(
        sim,
        modelname=gwfname,
        save_flows=True,
        newtonoptions="newton",
        model_nam_file=f"{gwfname}.nam",
    )

    # Instantiating MODFLOW 6 discretization package
    dis = flopy.mf6.ModflowGwfdis(
        gwf,
        nlay=nlayp,
        nrow=nrowp,
        ncol=ncolp,
        delr=delrp,
        delc=delcp,
        top=topp,
        botm=botmp,
        idomain=idomainp,
        filename=f"{gwfname}.dis",
    )

    # Instantiating MODFLOW 6 initial conditions package for flow model
    strt = [topp - 0.25, topp - 0.25, topp - 0.25]
    ic = flopy.mf6.ModflowGwfic(gwf, strt=strt, filename=f"{gwfname}.ic")

    # Instantiating MODFLOW 6 node-property flow package
    npf = flopy.mf6.ModflowGwfnpf(
        gwf,
        save_flows=False,
        alternative_cell_averaging="AMT-LMK",
        icelltype=icelltype,
        k=k11,
        k33=k33,
        save_specific_discharge=False,
        filename=f"{gwfname}.npf",
    )

    # Instantiating MODFLOW 6 output control package for flow model
    oc = flopy.mf6.ModflowGwfoc(
        gwf,
        budget_filerecord=f"{gwfname}.bud",
        head_filerecord=f"{gwfname}.hds",
        headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
        saverecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
        printrecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
    )

    # Instantiating MODFLOW 6 constant head package
    rowList = np.arange(0, nrowp).tolist()
    layList = np.arange(0, nlayp).tolist()
    chdspd_left = []
    chdspd_right = []

    # Loop through rows, the left & right sides will appear in separate,
    # dedicated packages
    hd_left = 49.75
    hd_right = 44.75
    for l in layList:
        for r in rowList:
            # first, do left side of model
            chdspd_left.append([(l, r, 0), hd_left])
            # finally, do right side of model
            chdspd_right.append([(l, r, ncolp - 1), hd_right])

    chdspd = {0: chdspd_left}
    chd1 = flopy.mf6.modflow.mfgwfchd.ModflowGwfchd(
        gwf,
        maxbound=len(chdspd),
        stress_period_data=chdspd,
        save_flows=False,
        pname="CHD-1",
        filename=f"{gwfname}.chd1.chd",
    )
    chdspd = {0: chdspd_right}
    chd2 = flopy.mf6.modflow.mfgwfchd.ModflowGwfchd(
        gwf,
        maxbound=len(chdspd),
        stress_period_data=chdspd,
        save_flows=False,
        pname="CHD-2",
        filename=f"{gwfname}.chd2.chd",
    )

    # Instantiating MODFLOW 6 Parent model's SFR package
    sfr = flopy.mf6.ModflowGwfsfr(
        gwf,
        print_stage=False,
        print_flows=False,
        budget_filerecord=gwfname + ".sfr.bud",
        save_flows=True,
        mover=True,
        pname="SFR-parent",
        time_conversion=86400.0,
        boundnames=False,
        nreaches=len(connsp),
        packagedata=pkdat,
        connectiondata=connsp,
        perioddata=sfrspd,
        filename=f"{gwfname}.sfr",
    )

    return sim, gwf

### Add child GWF model

In [ ]:
def add_child_gwf_model(sim, gwfnamec):
    # Leverage flopy's "Lgr" class; was imported at start of script
    ncpp = 3
    ncppl = [3, 3, 0]
    lgr = Lgr(
        nlayp,
        nrowp,
        ncolp,
        delrp,
        delcp,
        topp,
        botmp,
        idomainp,
        ncpp=ncpp,
        ncppl=ncppl,
        xllp=0.0,
        yllp=0.0,
    )

    # Get child grid info:
    delrc, delcc = lgr.get_delr_delc()
    idomainc = lgr.get_idomain()  # child idomain
    topc, botmc = lgr.get_top_botm()  # top/bottom of child grid

    # Instantiate MODFLOW 6 child gwf model
    gwfc = flopy.mf6.ModflowGwf(
        sim,
        modelname=gwfnamec,
        save_flows=True,
        newtonoptions="newton",
        model_nam_file=f"{gwfnamec}.nam",
    )

    # Instantiating MODFLOW 6 discretization package for the child model
    child_dis_shp = lgr.get_shape()
    nlayc = child_dis_shp[0]
    nrowc = child_dis_shp[1]
    ncolc = child_dis_shp[2]
    disc = flopy.mf6.ModflowGwfdis(
        gwfc,
        nlay=nlayc,
        nrow=nrowc,
        ncol=ncolc,
        delr=delrc,
        delc=delcc,
        top=topc,
        botm=botmc,
        idomain=idomainc,
        filename=f"{gwfnamec}.dis",
    )

    # Instantiating MODFLOW 6 initial conditions package for child model
    strtc = [
        topc - 0.25,
        topc - 0.25,
        topc - 0.25,
        topc - 0.25,
        topc - 0.25,
        topc - 0.25,
    ]
    icc = flopy.mf6.ModflowGwfic(gwfc, strt=strtc, filename=f"{gwfnamec}.ic")

    # Instantiating MODFLOW 6 node property flow package for child model
    icelltypec = [1, 1, 1, 0, 0, 0]
    npfc = flopy.mf6.ModflowGwfnpf(
        gwfc,
        save_flows=False,
        alternative_cell_averaging="AMT-LMK",
        icelltype=icelltypec,
        k=k11,
        k33=k33,
        save_specific_discharge=False,
        filename=f"{gwfnamec}.npf",
    )

    # Instantiating MODFLOW 6 output control package for the child model
    occ = flopy.mf6.ModflowGwfoc(
        gwfc,
        budget_filerecord=f"{gwfnamec}.bud",
        head_filerecord=f"{gwfnamec}.hds",
        headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
        saverecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
        printrecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
    )

    # Instantiating MODFLOW 6 Streamflow routing package for child model
    sfrc = flopy.mf6.ModflowGwfsfr(
        gwfc,
        print_stage=False,
        print_flows=False,
        budget_filerecord=gwfnamec + ".sfr.bud",
        save_flows=True,
        mover=True,
        pname="SFR-child",
        time_conversion=86400.00,
        boundnames=False,
        nreaches=len(connsc),
        packagedata=pkdatc,
        connectiondata=connsc,
        perioddata=sfrspdc,
        filename=f"{gwfnamec}.sfr",
    )

    return sim, gwfc, lgr

### Define function for building the simulation

In [ ]:
def build_models(sim_name, silent=False):
    # Instantiate the MODFLOW 6 simulation
    name = "lgr"
    sim_ws = workspace / sim_name
    sim = flopy.mf6.MFSimulation(
        sim_name=sim_name,
        version="mf6",
        sim_ws=sim_ws,
        exe_name="mf6",
        continue_=True,
    )

    # Instantiating MODFLOW 6 time discretization
    tdis_rc = []
    for i in range(len(perlen)):
        tdis_rc.append((perlen[i], nstp[i], tsmult[i]))
    flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=tdis_rc, time_units=time_units)

    # Instantiating MODFLOW 6 groundwater flow model
    gwfnamep = "gwf-" + name + "-parent"
    gwfnamec = "gwf-" + name + "-child"

    sim, gwfp = add_parent_gwf_model(sim, gwfnamep)

    # Instantiating MODFLOW 6 solver for the parent GWF model
    imsgwf = flopy.mf6.ModflowIms(
        sim,
        print_option="SUMMARY",
        outer_dvclose=hclose,
        outer_maximum=nouter,
        under_relaxation="NONE",
        inner_maximum=ninner,
        inner_dvclose=hclose,
        rcloserecord=rclose,
        linear_acceleration="BICGSTAB",
        scaling_method="NONE",
        reordering_method="NONE",
        relaxation_factor=relax,
        filename=f"{gwfnamep}.ims",
    )
    sim.register_ims_package(imsgwf, [gwfp.name])

    # -------------------------------
    # Next, add the child model
    # -------------------------------
    sim, gwfc, lgr = add_child_gwf_model(sim, gwfnamec)

    # Retrieve exchange data using Lgr class functionality
    exchange_data = lgr.get_exchange_data()

    # Establish MODFLOW 6 GWF-GWF exchange
    gwfgwf = flopy.mf6.ModflowGwfgwf(
        sim,
        exgtype="GWF6-GWF6",
        print_flows=True,
        print_input=True,
        exgmnamea=gwfnamep,
        exgmnameb=gwfnamec,
        nexg=len(exchange_data),
        exchangedata=exchange_data,
        mvr_filerecord=f"{name}.mvr",
        pname="EXG-1",
        filename=f"{name}.exg",
    )

    # Instantiate MVR package
    mvrpack = [[gwfnamep, "SFR-parent"], [gwfnamec, "SFR-child"]]
    maxpackages = len(mvrpack)

    # Set up static SFR-to-SFR connections that remain fixed for entire simulation
    static_mvrperioddata = [  # don't forget to use 0-based values
        [
            mvrpack[0][0],
            mvrpack[0][1],
            7,
            mvrpack[1][0],
            mvrpack[1][1],
            0,
            "FACTOR",
            1.0,
        ],
        [
            mvrpack[1][0],
            mvrpack[1][1],
            88,
            mvrpack[0][0],
            mvrpack[0][1],
            8,
            "FACTOR",
            1,
        ],
    ]

    mvrspd = {0: static_mvrperioddata}
    maxmvr = 2
    mvr = flopy.mf6.ModflowMvr(
        gwfgwf,
        modelnames=True,
        maxmvr=maxmvr,
        print_flows=True,
        maxpackages=maxpackages,
        packages=mvrpack,
        perioddata=mvrspd,
        filename=f"{name}.mvr",
    )

    return sim

### Execute building of the GWF model

In [ ]:
workspace = Path("./models") / example_name
sim = build_models(example_name, workspace)

### Write the simulation

In [ ]:
sim.write_simulation()

### Run the simulation

In [ ]:
sim.run_simulation()

### Define a function to plot the simulation output

In [ ]:
# Figure properties
figure_size = (7, 4)


def plot_results(mf6):
    sim_name = mf6.name
    #with styles.USGSPlot():
    # Start by retrieving some output
    mf6_out_pth = Path(mf6.simulation_data.mfpath.get_sim_path())
    sfr_parent_bud_file = next(iter(mf6.model_names)) + ".sfr.bud"
    sfr_child_bud_file = list(mf6.model_names)[1] + ".sfr.bud"
    sfr_parent_out = mf6_out_pth / sfr_parent_bud_file
    sfr_child_out = mf6_out_pth / sfr_child_bud_file
    modobjp = bf.CellBudgetFile(sfr_parent_out, precision="double")
    modobjc = bf.CellBudgetFile(sfr_child_out, precision="double")

    ckstpkper = modobjp.get_kstpkper()
    qp = []
    qc = []
    gwswp = []
    gwswc = []
    toMvrp = []
    fromMvrp = []
    toMvrc = []
    fromMvrc = []

    for kstpkper in ckstpkper:
        datp = modobjp.get_data(kstpkper=kstpkper, text="    FLOW-JA-FACE")
        datc = modobjc.get_data(kstpkper=kstpkper, text="    FLOW-JA-FACE")
        qp.append(datp)
        qc.append(datc)

        datp = modobjp.get_data(kstpkper=kstpkper, text="             GWF")
        datc = modobjc.get_data(kstpkper=kstpkper, text="             GWF")
        gwswp.append(datp[0])
        gwswc.append(datc[0])

        # No values for some reason?
        dat_fmp = modobjp.get_data(kstpkper=kstpkper, text="        FROM-MVR")
        dat_tmp = modobjp.get_data(kstpkper=kstpkper, text="          TO-MVR")
        toMvrp.append(dat_fmp[0])
        fromMvrp.append(dat_tmp[0])

        # No values for some reason?
        dat_fmc = modobjc.get_data(kstpkper=kstpkper, text="        FROM-MVR")
        dat_tmc = modobjc.get_data(kstpkper=kstpkper, text="          TO-MVR")
        toMvrc.append(dat_fmc[0])
        fromMvrc.append(dat_tmc[0])

    strmQ = np.zeros(len(connsp) + len(connsc))
    # Hardwire the first reach to the specified inflow rate
    strmQ[0] = 40.0
    for i, itm in enumerate(np.arange(1, len(qp[0][0]), 2)):
        # The first 8 reach of the parent SFR package are upstream of child grid
        if qp[0][0][itm][0] <= 8:
            strmQ[i + 1] = qp[0][0][itm][2]
        if i >= 8:
            break

    # The flow that is passed between the parent and child grids comes next
    strmQ[i] = dat_fmc[0][0][2]

    # Next, process the child grid
    for j, jtm in enumerate(np.arange(1, len(qc[0][0]), 2)):
        # process all reaches successively
        strmQ[i + (j + 1)] = qc[0][0][jtm][2]

    # The flow that is passed between the parent and child grids comes next
    from_mvr = next(itm[2] for itm in dat_fmp[0] if itm[2] > 0)
    strmQ[i + j + 2] = from_mvr

    # Finally, process the remaining parent model stream reaches
    for k, ktm in enumerate(np.arange(15, len(qp[0][0]), 2)):
        strmQ[i + j + 2 + (k + 1)] = qp[0][0][ktm][2]

    # Fill out a single vector of stream reach lengths
    all_rch_lengths = rlen[0:8] + rlenc + rlen[8:]
    # Now get center of all the reaches
    rch_lengths = []
    for i in np.arange(len(all_rch_lengths)):
        rch_lengths.append(np.sum(all_rch_lengths[0:i]) + (all_rch_lengths[i] / 2))

    # Make a continuous vector of the gw-sw exchanges
    gwsw_exg = np.zeros(len(connsp) + len(connsc))
    for i, itm in enumerate(gwswp[0]):
        if itm[0] <= 8:
            gwsw_exg[i] = itm[2]
        elif itm[0] > 8:
            gwsw_exg[(len(gwsw_exg) - (len(gwswp[0]) - i))] = itm[2]

    # Insert the child model gw/sw exchanges in their proper sequential spot
    for j, jtm in enumerate(gwswc[0]):
        gwsw_exg[8 + j] = jtm[2]

    fig, ax1 = plt.subplots(figsize=figure_size, dpi=150, tight_layout=True)
    pts = ax1.plot(rch_lengths, strmQ, "r^", label="Stream Flow", zorder=3)
    ax1.set_zorder(4)
    ax1.set_facecolor("none")
    ax1.text(
        rch_lengths[int(len(rch_lengths) / 2)],
        160,
        "Local Grid Refinement",
        ha="center",
        fontsize=7,
    )
    ax1.arrow(1080, 163, -440, 0, head_width=5, head_length=50, fc="k", ec="k")
    ax1.arrow(2150, 163, 395, 0, head_width=5, head_length=50, fc="k", ec="k")
    ax1.arrow(
        525,
        27,
        80,
        -7,
        head_width=5,
        head_length=50,
        fc="deeppink",
        ec="deeppink",
        linewidth=2,
    )  # deeppink
    ax1.arrow(
        2550,
        130,
        80,
        7,
        head_width=5,
        head_length=50,
        fc="deeppink",
        ec="deeppink",
        linewidth=2,
    )
    ax1.text(
        ((rch_lengths[7] + rch_lengths[8]) / 2) + 50,
        27,
        "First MVR\ntransfer",
        ha="left",
        fontsize=7,
    )
    ax1.text(
        (rch_lengths[7] + rch_lengths[8]) / 2 + 15,
        180 + 3.5,
        "Child Grid",
        rotation=90,
        ha="left",
        fontsize=7,
    )
    ax1.text(
        (rch_lengths[7] + rch_lengths[8]) / 2,
        180,
        "Parent Grid",
        rotation=90,
        ha="right",
        fontsize=7,
    )
    ax1.text(
        ((rch_lengths[96] + rch_lengths[97]) / 2) + 100,
        130,
        "Second MVR\ntransfer",
        ha="left",
        fontsize=7,
    )
    ax1.text(
        (rch_lengths[96] + rch_lengths[97]) / 2 + 15,
        180,
        "Parent Grid",
        rotation=90,
        ha="left",
        fontsize=7,
    )
    ax1.text(
        (rch_lengths[96] + rch_lengths[97]) / 2,
        180 + 3.5,
        "Child Grid",
        rotation=90,
        ha="right",
        fontsize=7,
    )
    ax1.set_xlim([0, 3500])
    ax1.set_ylim([-110, 250])
    ax1.set_xlabel("Distance Along River ($m$)")
    ax1.set_ylabel(r"Stream Flow ($\frac{m^3}{s}$)")
    ax1.vlines(
        x=(rch_lengths[7] + rch_lengths[8]) / 2, ymin=-60, ymax=235, linewidth=2
    )
    ax1.vlines(
        x=(rch_lengths[96] + rch_lengths[97]) / 2, ymin=-65, ymax=235, linewidth=2
    )

    ax2 = ax1.twinx()
    ax2.set_xlim([0, 3500])
    ax2.set_ylim([-11, 25])
    # [itm/10 for itm in all_rch_lengths]
    bar = ax2.bar(
        rch_lengths,
        gwsw_exg,
        align="center",
        color="b",
        width=[itm / 2 for itm in all_rch_lengths],
        label="GW-SW Exchange",
    )
    ax2.set_zorder(2)
    ax2.set_axisbelow(True)
    ax2.set_ylabel(
        r"Groundwater Surface-Water Exchange by Stream Reach ($\frac{m^3}{s}$)",
        rotation=270,
        labelpad=15,
    )
    ax2.yaxis.grid(color="silver", zorder=1)
    lns = pts + [bar]
    labs = [l.get_label() for l in lns]
    leg = ax2.legend(lns, labs, loc="lower right", frameon=True)
    leg.get_frame().set_linewidth(0.0)

    title = "River conditions with steady flow"
    letter = chr(ord("@") + 1)
    styles.heading(letter=letter, heading=title)

    plt.show()
    

In [ ]:
plot_results(sim)